# Configure DevWorkspace with AI Extensions

This notebook deploys a Dev Spaces workspace pre-configured with AI coding extensions (Continue + Roo Code) connected to the MaaS gateway. After completion, developers can open a browser and start coding with AI assistance immediately — no manual extension setup.

**What we'll do:**
1. Verify Dev Spaces operator is installed
2. Create extension ConfigMaps (Continue config + Roo Code provider profiles)
3. Deploy the DevWorkspace with ConfigMaps mounted
4. Verify workspace creation and extension configuration
5. Generate a Factory URL for team self-service


## 1. Verify Dev Spaces Operator

OpenShift Dev Spaces must be installed and the CheCluster instance must be running.


In [ ]:
%%bash
set +e
echo "=== Dev Spaces Operator ==="
oc get csv -n openshift-operators 2>/dev/null | grep devspaces || echo "⚠️  Dev Spaces operator not found"

echo ""
echo "=== CheCluster Instance ==="
oc get checluster -n openshift-devspaces 2>/dev/null || echo "⚠️  No CheCluster found — install Dev Spaces first"

echo ""
echo "=== Dev Spaces Route ==="
oc get route -n openshift-devspaces -l app=che 2>/dev/null | head -5 || echo "⚠️  Dashboard route not found"


## 2. Set Environment Variables

We need the cluster domain and MaaS API key to configure extensions. Update `MAAS_API_KEY` with a key generated in Phase 2.


In [ ]:
import subprocess, os

CLUSTER_DOMAIN = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
).stdout.strip()

MAAS_API_KEY = os.environ.get("MAAS_API_KEY", "REPLACE_WITH_MAAS_API_KEY")
NAMESPACE = "devspaces"

print(f"Cluster domain: {CLUSTER_DOMAIN}")
print(f"MaaS endpoint:  https://maas-api.{CLUSTER_DOMAIN}")
print(f"API key:        {'✅ Set' if MAAS_API_KEY != 'REPLACE_WITH_MAAS_API_KEY' else '⚠️  Using placeholder — update before applying'}")
print(f"Namespace:      {NAMESPACE}")


## 3. Deploy Continue Extension ConfigMap

Continue reads `~/.continue/config.yaml` at startup. This ConfigMap is mounted into the workspace pod at that path using DevWorkspace controller annotations.


In [ ]:
%%bash
set +e
# Substitute cluster domain in the manifest and apply
CLUSTER_DOMAIN=$(oc get ingresses.config cluster -o jsonpath='{.spec.domain}')

cat manifests/00-continue-config.yaml | \
  sed "s/CLUSTER_DOMAIN/${CLUSTER_DOMAIN}/g" | \
  oc apply -f -

echo ""
echo "=== ConfigMap created ==="
oc get configmap continue-config -n devspaces -o jsonpath='{.metadata.name}' && echo " ✅"


## 4. Deploy Roo Code Provider ConfigMap

Roo Code reads `provider_profiles.json` for API provider configuration. Critical: streaming must be **disabled** (`openAiStreamingEnabled: false`) because vLLM's streaming tool-call parser has known issues with Qwen3-Coder.


In [ ]:
%%bash
set +e
CLUSTER_DOMAIN=$(oc get ingresses.config cluster -o jsonpath='{.spec.domain}')

cat manifests/01-roo-code-config.yaml | \
  sed "s/CLUSTER_DOMAIN/${CLUSTER_DOMAIN}/g" | \
  oc apply -f -

echo ""
echo "=== ConfigMap created ==="
oc get configmap roo-code-provider-config -n devspaces -o jsonpath='{.metadata.name}' && echo " ✅"


## 5. Deploy DevWorkspace

The DevWorkspace CR creates a workspace pod with the universal developer image. ConfigMaps labeled with `controller.devfile.io/mount-to-devworkspace: "true"` are automatically mounted into the pod at the paths specified in their annotations.


In [ ]:
%%bash
set +e
CLUSTER_DOMAIN=$(oc get ingresses.config cluster -o jsonpath='{.spec.domain}')

cat manifests/02-devworkspace.yaml | \
  sed "s/CLUSTER_DOMAIN/${CLUSTER_DOMAIN}/g" | \
  oc apply -f -

echo ""
echo "=== DevWorkspace ==="
oc get devworkspace -n devspaces


## 6. Verify Workspace and Extensions

Check that the workspace pod is running and ConfigMaps are mounted at the expected paths.


In [ ]:
%%bash
set +e
echo "=== Workspace Pod Status ==="
oc get pods -n devspaces -l controller.devfile.io/devworkspace_name=ai-coding-workspace

WS_POD=$(oc get pods -n devspaces -l controller.devfile.io/devworkspace_name=ai-coding-workspace -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -n "$WS_POD" ]; then
  echo ""
  echo "=== Continue Config Mounted? ==="
  oc exec -n devspaces $WS_POD -- cat /home/user/.continue/config.yaml 2>/dev/null | head -5 || echo "Not yet mounted (workspace may still be starting)"

  echo ""
  echo "=== Roo Code Config Mounted? ==="
  oc exec -n devspaces $WS_POD -- cat /checode/remote/data/User/globalStorage/rooveterinaryinc.roo-cline/settings/provider_profiles.json 2>/dev/null | head -5 || echo "Not yet mounted"
else
  echo "Workspace pod not yet running — wait for DevWorkspace to start"
fi


## 7. Factory URL for Team Self-Service

Share this URL with your team — clicking it creates a pre-configured workspace automatically.


In [ ]:
import subprocess

cluster_domain = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
).stdout.strip()

factory_url = f"https://devspaces.{cluster_domain}/dashboard/#/devspaces/createWorkspace/devfile?url=https://github.com/hyogrin/rhoai-code-assistant-lab"

print("=== Factory URL ===")
print(f"\n{factory_url}")
print("\nShare this URL with developers — they authenticate with OpenShift")
print("and receive a workspace with Continue + Roo Code pre-configured.")


## Summary

| Component | Status |
|-----------|--------|
| Dev Spaces operator | Verified |
| Continue ConfigMap | Deployed (auto-mounted to `~/.continue/config.yaml`) |
| Roo Code ConfigMap | Deployed (auto-mounted to provider_profiles.json path) |
| DevWorkspace | Created |
| Factory URL | Generated |

**Extension behavior:**
- **Continue**: Tab autocomplete + chat, fully automated via config.yaml
- **Roo Code**: Multi-mode agent (Code/Architect/Debug), non-streaming for tool-call compatibility
- **Cline**: Not pre-configured — developers who want Cline set it up manually via the extension UI

→ Continue to `3_test_extensions.ipynb` to verify that extensions can reach the MaaS gateway and tool calling works correctly.
